# Hacker News Tech Engagement Analysis

## Project Objective

The goal of this project is to collect and analyze technology news data from Hacker News in order to identify which stories and recurring sources generate the highest reader engagement.

This notebook demonstrates a compact end-to-end analyst workflow:

**Web extraction → cleaning → validation → feature engineering → analysis → reporting**


## Tools

- Python
- Requests
- BeautifulSoup
- Pandas
- Matplotlib
- XlsxWriter

The project uses only three Hacker News pages so the scope remains focused and recruiter-friendly.


In [ ]:
# Import the required libraries
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import matplotlib.pyplot as plt

# Colab-only helper for automatic file download
try:
    from google.colab import files
    RUNNING_IN_COLAB = True
except ImportError:
    RUNNING_IN_COLAB = False


## 1. Web Data Extraction

Hacker News stores the visible story information and its metadata in two separate sibling table rows.

The first row contains the rank, title, link, and source.  
The next row contains the score, author, age, and comments.

The scraper connects those two rows into one analytical record.


In [ ]:
base_url = "https://news.ycombinator.com/news"
stories_data = []

for page in range(1, 4):

    print(f"Processing page {page}...")

    response = requests.get(
        base_url,
        params={"p": page},
        timeout=30
    )

    if response.status_code != 200:
        print(f"Error accessing page {page}")
        continue

    soup = BeautifulSoup(response.text, "html.parser")
    stories = soup.select("tr.athing")

    for story in stories:

        story_id = story.get("id")

        rank_element = story.select_one("span.rank")
        rank = (
            rank_element.get_text(strip=True).replace(".", "")
            if rank_element
            else None
        )

        title_element = story.select_one("span.titleline > a")

        if title_element:
            title = title_element.get_text(strip=True)
            link = title_element.get("href")
        else:
            title = None
            link = None

        source_element = story.select_one("span.sitestr")
        source = (
            source_element.get_text(strip=True)
            if source_element
            else "news.ycombinator.com"
        )

        metadata_row = story.find_next_sibling("tr")

        score_element = (
            metadata_row.select_one("span.score")
            if metadata_row
            else None
        )

        if score_element:
            score = (
                score_element
                .get_text(strip=True)
                .replace(" points", "")
                .replace(" point", "")
            )
        else:
            score = 0

        author_element = (
            metadata_row.select_one("a.hnuser")
            if metadata_row
            else None
        )

        author = (
            author_element.get_text(strip=True)
            if author_element
            else "Unknown"
        )

        age_element = (
            metadata_row.select_one("span.age")
            if metadata_row
            else None
        )

        age = (
            age_element.get_text(strip=True)
            if age_element
            else None
        )

        comments = 0

        if metadata_row:
            for link_element in metadata_row.select("a"):

                text = link_element.get_text(strip=True)

                if "comment" in text:
                    comments = text.split()[0]

                elif text == "discuss":
                    comments = 0

        stories_data.append({
            "Story_ID": story_id,
            "Rank": rank,
            "Title": title,
            "Source": source,
            "Score": score,
            "Author": author,
            "Age": age,
            "Comments": comments,
            "Link": link,
            "Page": page
        })

    time.sleep(1)

print(f"\nTotal stories collected: {len(stories_data)}")


## 2. Data Transformation

The extracted records are converted into a Pandas DataFrame. Numeric fields are converted to numeric data types so they can be analyzed correctly.


In [ ]:
df = pd.DataFrame(stories_data)

numeric_columns = [
    "Rank",
    "Score",
    "Comments",
    "Page"
]

for column in numeric_columns:
    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )

df.head()


## 3. Data Quality Validation

Before analysis, the dataset is checked for missing values and duplicate story IDs.


In [ ]:
print("Dataset dimensions:")
print(df.shape)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate Story IDs:")
print(df["Story_ID"].duplicated().sum())


## 4. Feature Engineering

A simple custom engagement metric is created:

**Engagement = Score + Comments**

This is not an official Hacker News metric. It is used only for this portfolio analysis to compare overall interaction across stories.


In [ ]:
df["Engagement"] = (
    df["Score"].fillna(0)
    +
    df["Comments"].fillna(0)
)

df.head()


## 5. Analysis

The analysis focuses on two recruiter-friendly questions:

1. Which stories generated the most engagement?
2. Which recurring sources generated the strongest average engagement?


In [ ]:
top_stories = (
    df[
        [
            "Title",
            "Source",
            "Score",
            "Comments",
            "Engagement"
        ]
    ]
    .sort_values(
        "Engagement",
        ascending=False
    )
    .head(10)
)

top_stories


In [ ]:
source_analysis = (
    df.groupby("Source")
    .agg(
        Stories=("Title", "count"),
        Average_Score=("Score", "mean"),
        Average_Comments=("Comments", "mean"),
        Average_Engagement=("Engagement", "mean")
    )
    .reset_index()
)

source_analysis = (
    source_analysis[
        source_analysis["Stories"] >= 2
    ]
    .sort_values(
        "Average_Engagement",
        ascending=False
    )
)

source_analysis.head(10)


## 6. Visualizations

Only two charts are used so the project remains concise and easy to review.


In [ ]:
top_10 = (
    df.sort_values(
        "Engagement",
        ascending=False
    )
    .head(10)
    .sort_values("Engagement")
)

plt.figure(figsize=(10, 6))
plt.barh(
    top_10["Title"],
    top_10["Engagement"]
)
plt.xlabel("Engagement")
plt.ylabel("Story")
plt.title("Top 10 Hacker News Stories by Engagement")
plt.tight_layout()
plt.show()


In [ ]:
top_sources_chart = (
    df["Source"]
    .value_counts()
    .head(10)
    .sort_values()
)

plt.figure(figsize=(9, 5))
plt.barh(
    top_sources_chart.index,
    top_sources_chart.values
)
plt.xlabel("Number of Stories")
plt.ylabel("Source")
plt.title("Most Frequent Sources on Hacker News")
plt.tight_layout()
plt.show()


## 7. Export Final Report

The project exports:

- A clean CSV dataset.
- A formatted Excel workbook with three sheets:
  - `Stories`
  - `Top Stories`
  - `Source Analysis`

The workbook includes Excel tables, filters, frozen headers, adjusted column widths, and engagement data bars.

When run in Google Colab, the Excel report downloads automatically.


In [ ]:
csv_file = "hacker_news_engagement.csv"
excel_file = "hacker_news_engagement_analysis.xlsx"

df.to_csv(
    csv_file,
    index=False,
    encoding="utf-8-sig"
)

with pd.ExcelWriter(
    excel_file,
    engine="xlsxwriter"
) as writer:

    df.to_excel(
        writer,
        sheet_name="Stories",
        index=False
    )

    top_stories.to_excel(
        writer,
        sheet_name="Top Stories",
        index=False
    )

    source_analysis.to_excel(
        writer,
        sheet_name="Source Analysis",
        index=False
    )

    workbook = writer.book

    integer_format = workbook.add_format({
        "num_format": "0",
        "align": "center"
    })

    decimal_format = workbook.add_format({
        "num_format": "0.00"
    })

    worksheet = writer.sheets["Stories"]
    rows, columns = df.shape

    worksheet.add_table(
        0,
        0,
        rows,
        columns - 1,
        {
            "name": "StoriesTable",
            "style": "Table Style Medium 2",
            "columns": [
                {"header": column}
                for column in df.columns
            ]
        }
    )

    worksheet.freeze_panes(1, 0)
    worksheet.set_column("A:A", 14)
    worksheet.set_column("B:B", 8, integer_format)
    worksheet.set_column("C:C", 65)
    worksheet.set_column("D:D", 25)
    worksheet.set_column("E:E", 10, integer_format)
    worksheet.set_column("F:F", 18)
    worksheet.set_column("G:G", 15)
    worksheet.set_column("H:H", 12, integer_format)
    worksheet.set_column("I:I", 55)
    worksheet.set_column("J:J", 8, integer_format)
    worksheet.set_column("K:K", 14, integer_format)

    engagement_column = df.columns.get_loc(
        "Engagement"
    )

    worksheet.conditional_format(
        1,
        engagement_column,
        rows,
        engagement_column,
        {"type": "data_bar"}
    )

    worksheet_top = writer.sheets[
        "Top Stories"
    ]

    top_rows, top_columns = top_stories.shape

    worksheet_top.add_table(
        0,
        0,
        top_rows,
        top_columns - 1,
        {
            "name": "TopStoriesTable",
            "style": "Table Style Medium 4",
            "columns": [
                {"header": column}
                for column in top_stories.columns
            ]
        }
    )

    worksheet_top.freeze_panes(1, 0)
    worksheet_top.set_column("A:A", 70)
    worksheet_top.set_column("B:B", 25)
    worksheet_top.set_column("C:E", 15)

    engagement_top_col = (
        top_stories.columns.get_loc(
            "Engagement"
        )
    )

    worksheet_top.conditional_format(
        1,
        engagement_top_col,
        top_rows,
        engagement_top_col,
        {"type": "data_bar"}
    )

    worksheet_sources = writer.sheets[
        "Source Analysis"
    ]

    source_rows, source_columns = (
        source_analysis.shape
    )

    worksheet_sources.add_table(
        0,
        0,
        source_rows,
        source_columns - 1,
        {
            "name": "SourceAnalysisTable",
            "style": "Table Style Medium 9",
            "columns": [
                {"header": column}
                for column in source_analysis.columns
            ]
        }
    )

    worksheet_sources.freeze_panes(1, 0)
    worksheet_sources.set_column("A:A", 30)
    worksheet_sources.set_column("B:B", 12)
    worksheet_sources.set_column(
        "C:E",
        20,
        decimal_format
    )

print("✅ Excel report created successfully.")
print(f"📄 File: {excel_file}")
print(f"📊 Stories analyzed: {len(df)}")

if RUNNING_IN_COLAB:
    print("⬇️ Starting automatic download...")
    files.download(excel_file)
else:
    print("The Excel file was saved in the current working directory.")


## Portfolio Takeaway

This project demonstrates more than HTML extraction. It shows how raw web data can be transformed into a small analytical product with quality checks, business-oriented metrics, visual analysis, and automated reporting.
